# Model Explorer

Load a trained model from `logs/` and run it on train/validation data.

**Usage:** Set `LOG_PREFIX` below to match the log files you want to load (e.g. `"mlx_smoke"`).
The notebook will:
1. Extract the model definition from the log `.txt` file (which contains the training script)
2. Load weights from the quantized `.int8.ptz` checkpoint (or raw `.npz`)
3. Load train and validation data using baseline data loading utilities
4. Let you run the model on examples and inspect predictions

In [ ]:
# === CONFIGURATION ===
# Set this to the prefix of your log files (without extension).
# Available logs can be found with: !ls logs/*.txt
LOG_PREFIX = "mlx_smoke"

# Whether to load the quantized (int8) or raw (fp32) checkpoint
USE_QUANTIZED = True

# Sequence length for data batches
SEQ_LEN = 1024

# Data paths (defaults match the baseline)
DATA_PATH = "./data/datasets/fineweb10B_sp1024"
TOKENIZER_PATH = "./data/tokenizers/fineweb_1024_bpe.model"

In [ ]:
import glob
import importlib.util
import os
import pickle
import sys
import tempfile
import zlib
from pathlib import Path

import numpy as np
import sentencepiece as spm

import mlx.core as mx
import mlx.nn as nn
from mlx.utils import tree_flatten, tree_unflatten

## 1. Load the training script from the log file

The `.txt` log files contain the full training script followed by training output.
We extract just the Python source, write it to a temp file, and import it as a module.
This gives us the exact model class and hyperparameters that were used for training.

In [ ]:
log_dir = Path("logs")
log_txt = log_dir / f"{LOG_PREFIX}.txt"
assert log_txt.exists(), f"Log file not found: {log_txt}"

# Determine checkpoint path
if USE_QUANTIZED:
    ckpt_path = log_dir / f"{LOG_PREFIX}_mlx_model.int8.ptz"
else:
    ckpt_path = log_dir / f"{LOG_PREFIX}_mlx_model.npz"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

print(f"Log file:   {log_txt}")
print(f"Checkpoint: {ckpt_path} ({ckpt_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Extract the Python source from the log file.
# The log format is: full training script, then training output lines.
# We find where the script ends by looking for the if __name__ block and its body.
raw_lines = log_txt.read_text().splitlines()

# Find the end of the Python source: after `if __name__` block, the first line
# that looks like training output (e.g. "step:" or non-Python) marks the boundary.
in_main = False
source_end = len(raw_lines)
for i, line in enumerate(raw_lines):
    if line.strip().startswith("if __name__"):
        in_main = True
    if in_main and i > 0:
        # Training output lines start with known prefixes or are unindented non-Python
        stripped = line.strip()
        if stripped and not stripped.startswith("#") and ":" in stripped:
            # Check if it looks like a log line (e.g. "step:1/200" or "saved_model:...")
            first_token = stripped.split(":")[0].split("(")[0].split("[")[0]
            if first_token in (
                "step",
                "val_progress",
                "saved_model",
                "serialized_model_int8_zlib",
                "final_int8_zlib_roundtrip",
                "final_int8_zlib_roundtrip_exact",
                "stopping_early",
                "WARNING",
            ):
                source_end = i
                break

script_source = "\n".join(raw_lines[:source_end])
print(f"Extracted {source_end} lines of Python source from {log_txt.name}")

In [ ]:
# Import the training script as a module (without running __main__)
# We strip the if __name__ == "__main__" block to avoid executing training.
main_idx = None
for i, line in enumerate(raw_lines[:source_end]):
    if line.strip().startswith("if __name__"):
        main_idx = i
        break

importable_source = "\n".join(raw_lines[:main_idx]) if main_idx else script_source

# Write to a temp file and import
_tmpdir = tempfile.mkdtemp()
_tmp_path = os.path.join(_tmpdir, f"{LOG_PREFIX.replace('-', '_')}_module.py")
with open(_tmp_path, "w") as f:
    f.write(importable_source)

spec = importlib.util.spec_from_file_location("train_module", _tmp_path)
train_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_module)

print(
    f"Imported module with attributes: {[a for a in dir(train_module) if not a.startswith('_')][:20]}"
)

## 2. Instantiate the model and load weights

In [ ]:
# Get hyperparameters from the imported module
hp = train_module.Hyperparameters()
print("Hyperparameters:")
for field in [
    "vocab_size",
    "num_layers",
    "model_dim",
    "num_heads",
    "num_kv_heads",
    "mlp_mult",
    "tie_embeddings",
    "logit_softcap",
    "rope_base",
]:
    if hasattr(hp, field):
        print(f"  {field}: {getattr(hp, field)}")

In [ ]:
# Build the model using the same constructor args from the training script.
# The GPT class signature can vary between records, so we introspect.
import inspect

GPT = train_module.GPT
sig = inspect.signature(GPT.__init__)
init_params = [p for p in sig.parameters if p != "self"]
print(f"GPT.__init__ params: {init_params}")

# Map from common init param names to hyperparameter fields
param_map = {
    "vocab_size": hp.vocab_size,
    "num_layers": hp.num_layers,
    "dim": hp.model_dim,
    "num_heads": hp.num_heads,
    "num_kv_heads": hp.num_kv_heads,
    "mlp_mult": hp.mlp_mult,
    "logit_chunk_tokens": getattr(hp, "logit_chunk_tokens", 0),
    "logit_softcap": getattr(hp, "logit_softcap", 30.0),
    "rope_base": getattr(hp, "rope_base", 10000.0),
    "tied_embed_init_std": getattr(hp, "tied_embed_init_std", 0.005),
    "qk_gain_init": getattr(hp, "qk_gain_init", 1.5),
}

# Build kwargs from what the constructor actually accepts
kwargs = {}
for p in init_params:
    if p in param_map:
        kwargs[p] = param_map[p]
    else:
        # Try getting it from hyperparameters directly
        if hasattr(hp, p):
            kwargs[p] = getattr(hp, p)
        else:
            print(f"  WARNING: unknown init param '{p}' — using default")

print(f"\nConstructing GPT with: {kwargs}")
model = GPT(**kwargs)
print(f"Model created successfully.")

In [ ]:
# Load checkpoint weights
if USE_QUANTIZED:
    with open(ckpt_path, "rb") as f:
        quant_blob = f.read()
    quant_obj = pickle.loads(zlib.decompress(quant_blob))
    flat_state = train_module.dequantize_state_dict_int8(quant_obj)
    print(f"Loaded quantized checkpoint: {len(flat_state)} tensors")
else:
    raw = dict(np.load(str(ckpt_path)))
    flat_state = {k: mx.array(v) for k, v in raw.items()}
    print(f"Loaded raw checkpoint: {len(flat_state)} tensors")

model.update(tree_unflatten(list(flat_state.items())))
mx.eval(model.parameters())

# Count parameters
num_params = sum(v.size for _, v in tree_flatten(model.parameters()))
print(f"Model parameters: {num_params:,}")

## 3. Load tokenizer and data

In [ ]:
# Load tokenizer
sp = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)
print(f"Tokenizer loaded: vocab_size={sp.vocab_size()}")

In [ ]:
# Use the baseline data loading utilities
# We import from train_gpt_mlx.py directly for the data loading parts
sys.path.insert(0, str(Path(".").resolve()))
from train_gpt_mlx import (
    load_data_shard,
    load_validation_tokens,
    TokenStream,
    TokenLoader,
    build_sentencepiece_luts,
)

# Load validation tokens
val_pattern = f"{DATA_PATH}/fineweb_val_*.bin"
val_tokens = load_validation_tokens(val_pattern, seq_len=SEQ_LEN)
print(
    f"Validation tokens: {val_tokens.shape[0]:,} (enough for {(val_tokens.shape[0] - 1) // SEQ_LEN} sequences)"
)

# Set up train data loader
train_pattern = f"{DATA_PATH}/fineweb_train_*.bin"
train_loader = TokenLoader(train_pattern, dataset_name="train")
print(f"Train loader ready.")

# Build byte counting LUTs for BPB calculation
base_bytes_lut, has_leading_space_lut, is_boundary_token_lut = build_sentencepiece_luts(
    sp, hp.vocab_size
)

## 4. Run the model on examples

In [ ]:
def get_val_batch(batch_idx: int = 0, seq_len: int = SEQ_LEN):
    """Get a batch from validation data by index."""
    start = batch_idx * seq_len
    end = start + seq_len + 1
    if end > val_tokens.shape[0]:
        raise IndexError(f"batch_idx {batch_idx} out of range")
    chunk = val_tokens[start:end]
    x = mx.array(chunk[:-1].reshape(1, seq_len), dtype=mx.int32)
    y = mx.array(chunk[1:].reshape(1, seq_len), dtype=mx.int32)
    return x, y


def get_train_batch(num_tokens: int = SEQ_LEN, seq_len: int = SEQ_LEN):
    """Get the next batch from the training stream."""
    return train_loader.next_batch(num_tokens, seq_len)


def decode_tokens(token_ids):
    """Decode token IDs to text."""
    if isinstance(token_ids, mx.array):
        token_ids = token_ids.tolist()
    if isinstance(token_ids, np.ndarray):
        token_ids = token_ids.tolist()
    # Flatten if needed
    if isinstance(token_ids[0], list):
        token_ids = token_ids[0]
    return sp.decode(token_ids)


def compute_loss(x, y):
    """Compute cross-entropy loss for a single (x, y) pair."""
    loss = model.loss(x, y)
    mx.eval(loss)
    return float(loss)


print(
    "Helper functions defined: get_val_batch, get_train_batch, decode_tokens, compute_loss"
)

In [ ]:
# === Example: look at a validation sequence and model predictions ===
x, y = get_val_batch(0)

# Compute loss
loss = compute_loss(x, y)
print(f"Loss on val batch 0: {loss:.4f}")
print(f"Perplexity: {np.exp(loss):.2f}")
print()

# Show the input text (first 500 chars)
input_text = decode_tokens(x)
print("=== Input text (first 500 chars) ===")
print(input_text[:500])
print("...")

In [ ]:
# === Example: get model predictions and compare with ground truth ===
x, y = get_val_batch(0)

# Forward pass to get hidden states, then compute logits
hidden = model(x)  # (1, seq_len, dim)
# Logits via tied embeddings
logits = (
    hidden.reshape(-1, model.tok_emb.weight.shape[1])
    @ model.tok_emb.weight.astype(hidden.dtype).T
)
if hasattr(model, "softcap"):
    logits = model.softcap(logits)
mx.eval(logits)

# Get top-k predictions for each position
probs = mx.softmax(logits.astype(mx.float32), axis=-1)
mx.eval(probs)

# Show predictions for a few positions
K = 5
positions = [0, 10, 50, 100, 200]
y_flat = y.reshape(-1)
print(f"Length of sequence: {y_flat.shape[0]}")

for pos in positions:
    if pos >= logits.shape[0]:
        break
    pos_probs = np.array(probs[pos])
    top_k_ids = np.argsort(pos_probs)[-K:][::-1]
    actual_id = int(y_flat[pos])
    actual_prob = float(pos_probs[actual_id])

    print(f"\n--- Position {pos} ---")
    print(f"  Context: ...{decode_tokens(x[0, max(0, pos - 5) : pos + 1].tolist())}")
    print(
        f"  Actual next: '{sp.id_to_piece(actual_id)}' (id={actual_id}, prob={actual_prob:.4f})"
    )
    print(f"  Top-{K} predictions:")
    for rank, tid in enumerate(top_k_ids):
        tid = int(tid)
        print(
            f"    {rank + 1}. '{sp.id_to_piece(tid)}' (id={tid}, prob={float(pos_probs[tid]):.4f})"
            + (" <<<" if tid == actual_id else "")
        )

In [ ]:
# === Example: greedy autoregressive generation from a prompt ===
def generate(prompt_tokens, max_new_tokens=100, temperature=0.8, top_k=50):
    """Simple autoregressive generation."""
    if isinstance(prompt_tokens, list):
        tokens = list(prompt_tokens)
    else:
        tokens = (
            prompt_tokens.tolist()
            if hasattr(prompt_tokens, "tolist")
            else list(prompt_tokens)
        )

    for _ in range(max_new_tokens):
        # Use last SEQ_LEN tokens as context
        ctx = tokens[-SEQ_LEN:]
        x = mx.array([ctx], dtype=mx.int32)
        hidden = model(x)
        # Get logits for the last position
        last_hidden = hidden[0, -1, :]  # (dim,)
        logits = last_hidden @ model.tok_emb.weight.astype(hidden.dtype).T
        if hasattr(model, "softcap"):
            logits = model.softcap(logits)
        logits = logits.astype(mx.float32)

        if temperature <= 0:
            # Greedy
            next_id = int(mx.argmax(logits))
        else:
            # Top-k sampling
            logits_np = np.array(logits)
            if top_k > 0 and top_k < logits_np.shape[0]:
                indices_to_remove = logits_np < np.partition(logits_np, -top_k)[-top_k]
                logits_np[indices_to_remove] = -float("inf")
            logits_np = logits_np / temperature
            logits_np -= logits_np.max()
            probs = np.exp(logits_np)
            probs /= probs.sum()
            next_id = int(np.random.choice(len(probs), p=probs))

        tokens.append(next_id)

    return tokens


# Generate from the start of a validation sequence
x, y = get_val_batch(5)
prompt_len = 50  # Use first 50 tokens as prompt
prompt = x[0, :prompt_len].tolist()

print("=== Prompt ===")
print(decode_tokens(prompt))
print()

generated = generate(prompt, max_new_tokens=100, temperature=0.8)
print("=== Generated continuation ===")
print(decode_tokens(generated[prompt_len:]))
print()

print("=== Actual continuation ===")
print(decode_tokens(y[0, prompt_len - 1 : prompt_len + 99].tolist()))

In [ ]:
# === Example: compare loss on a few train vs val batches ===
print("Validation losses:")
for i in range(5):
    x, y = get_val_batch(i)
    loss = compute_loss(x, y)
    print(f"  val batch {i}: loss={loss:.4f}  ppl={np.exp(loss):.2f}")

print("\nTraining losses:")
for i in range(5):
    x, y = get_train_batch()
    loss = compute_loss(x, y)
    print(f"  train batch {i}: loss={loss:.4f}  ppl={np.exp(loss):.2f}")